# Creación y preparación de la BBDD
### 1. Creación bbdd desde csv
La query de SPARQL nos devuelve un .csv, obtenemos la bbdd a partir de ese fichero.

In [2]:
import sqlite3
import pandas as pd

In [ ]:

db_path = r".\artworks_raw.db"
csv_path = r".\query.csv"

# Establecemos canal de comunicación con la bbdd
connection = sqlite3.connect(db_path)
dataFrame = pd.read_csv(csv_path)
dataFrame.to_sql("artworks_raw", connection, if_exists='replace', index=True, index_label="id")

# Un cursor ejecuta comandos SQL y recoge los resultados.
db_cursor = connection.cursor()
for row in db_cursor.execute("SELECT * FROM artworks_raw"):
    print(row)

# Al abrir la conexión, se lockea el fichero para evitar problemas de concurrencia
# Hay que cerarla para liberar el lock y los recursos del sistema
connection.close()

(0, 'http://www.wikidata.org/entity/Q9074362', 'Santo Domingo de Silos (Bartolomé Bermejo)', 'http://www.wikidata.org/entity/Q3821456', 'Martín Bernat', 'http://www.wikidata.org/entity/Q4692', 'Renacimiento', 'http://www.wikidata.org/entity/Q160112', 'Museo del Prado', 'http://commons.wikimedia.org/wiki/Special:FilePath/Santo%20Domingo%20de%20Silos%20entronizado%20como%20obispo%2C%20por%20Bartolom%C3%A9%20Bermejo.jpg')
(1, 'http://www.wikidata.org/entity/Q27697128', 'Fernando I de Castilla acogiendo a Santo Domingo de Silos', 'http://www.wikidata.org/entity/Q3821456', 'Martín Bernat', 'http://www.wikidata.org/entity/Q4692', 'Renacimiento', 'http://www.wikidata.org/entity/Q160112', 'Museo del Prado', 'http://commons.wikimedia.org/wiki/Special:FilePath/Bernat-bermejo%20fernando%20I%20castilla.jpg')
(2, 'http://www.wikidata.org/entity/Q61911383', 'Altarpiece of Santo Domingo de Silos', 'http://www.wikidata.org/entity/Q3821456', 'Martín Bernat', 'http://www.wikidata.org/entity/Q4692', 'Ren

### 2. Creación de bbdd "limpia"
La bbdd obtenida contiene las referencias SPARQL a los elementos en wikidata. Por claridad de la bbdd queremos eliminar estas columnas, pero mantenerlas en una bbdd aparte por trazabilidad de los elementos.

El cuadro "El gran masturbador" de Dalí tiene una foto no apropiada para el posterior tratamiento y reconocimiento de las imágenes, así que eliminamos esa fila.
El guernica está dos veces

In [ ]:
# Eliminamos las filas defectuosas ANTES  de hacer el backup
raw_db_path = r".\artworks_raw.db"
conn = sqlite3.connect(db_path)
db_cursor = conn.cursor()
db_cursor.execute("DELETE FROM artworks_raw WHERE paintingLabel = 'El gran masturbador'") # El gran masturbador
db_cursor.execute("DELETE FROM artworks_raw WHERE id = 67") # Guernica duplicado
conn.commit()
conn.close()

In [ ]:
raw_db_path = r".\artworks_raw.db"
clean_db_path = r".\artworks.db"

source_conn = sqlite3.connect(raw_db_path)
dest_conn = sqlite3.connect(clean_db_path)

try:
    # Hacemos la copia (como backup)
    with dest_conn:
        source_conn.backup(dest_conn)
    print("Backup completed successfully!")
    
except sqlite3.Error as error:
    print("Error while taking backup:", error)
    
finally:
    # The table in the new database is keeping its old name
    # We change it
    cursor = dest_conn.cursor()
    cursor.execute("ALTER TABLE artworks_raw RENAME TO artworks;")
    source_conn.close()
    dest_conn.close()

Backup completed successfully!


In [ ]:
# Eliminamos columnas con referencias
clean_db_path = r".\artworks.db"
conn = sqlite3.connect(clean_db_path)
db_cursor = conn.cursor()
# SQLite does not support dropping multiple columns in a single ALTER statement
db_cursor.execute("ALTER TABLE artworks DROP COLUMN painting;")
db_cursor.execute("ALTER TABLE artworks RENAME COLUMN paintingLabel TO painting")
db_cursor.execute("ALTER TABLE artworks DROP COLUMN creator;")
db_cursor.execute("ALTER TABLE artworks RENAME COLUMN creatorLabel TO creator")
db_cursor.execute("ALTER TABLE artworks DROP COLUMN movement;")
db_cursor.execute("ALTER TABLE artworks RENAME COLUMN movementLabel TO movement")
db_cursor.execute("ALTER TABLE artworks DROP COLUMN museum;")
db_cursor.execute("ALTER TABLE artworks RENAME COLUMN museumLabel TO museum")
# Para asegurarnos de que los cambios se aplican a la base de datos hacemos commit
conn.commit()

for row in db_cursor.execute("SELECT * FROM artworks"):
    print(row)
conn.close()

(0, 'Santo Domingo de Silos (Bartolomé Bermejo)', 'Martín Bernat', 'Renacimiento', 'Museo del Prado', 'http://commons.wikimedia.org/wiki/Special:FilePath/Santo%20Domingo%20de%20Silos%20entronizado%20como%20obispo%2C%20por%20Bartolom%C3%A9%20Bermejo.jpg')
(1, 'Fernando I de Castilla acogiendo a Santo Domingo de Silos', 'Martín Bernat', 'Renacimiento', 'Museo del Prado', 'http://commons.wikimedia.org/wiki/Special:FilePath/Bernat-bermejo%20fernando%20I%20castilla.jpg')
(2, 'Altarpiece of Santo Domingo de Silos', 'Martín Bernat', 'Renacimiento', 'Museo del Prado', 'http://commons.wikimedia.org/wiki/Special:FilePath/B.Bermejo%20ret.St.Domingo%20Silos%20hipotesi%204130.jpg')
(3, 'La lechera de Burdeos', 'Francisco de Goya', 'neoclasicismo', 'Museo del Prado', 'http://commons.wikimedia.org/wiki/Special:FilePath/Goya%20MilkMaid.jpg')
(4, 'Los duques de Osuna y sus hijos', 'Francisco de Goya', 'neoclasicismo', 'Museo del Prado', 'http://commons.wikimedia.org/wiki/Special:FilePath/Los%20duques%2

### 3. Obtención de imágenes en formato BLOB
Para poder pasarle las imágenes al modelo de vectorización, necesitamos guardarlas en la bbdd en formato BLOB

In [ ]:
import sqlite3
import requests
db_path = r".\artworks.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
# Añadimos la nueva columna
cursor.execute("ALTER TABLE artworks ADD COLUMN image_blob BLOB")
conn.close()

In [1]:
import pywikibot
import sqlite3
import os
import requests
import urllib
import time
db_path = r".\artworks.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

In [2]:
cursor.execute("SELECT id FROM artworks")
id_list = cursor.fetchall()
id_list

[(0,),
 (1,),
 (2,),
 (3,),
 (4,),
 (5,),
 (6,),
 (7,),
 (8,),
 (9,),
 (10,),
 (11,),
 (12,),
 (13,),
 (14,),
 (15,),
 (16,),
 (17,),
 (18,),
 (19,),
 (20,),
 (21,),
 (22,),
 (23,),
 (24,),
 (25,),
 (26,),
 (27,),
 (28,),
 (29,),
 (30,),
 (31,),
 (32,),
 (33,),
 (34,),
 (35,),
 (36,),
 (37,),
 (38,),
 (39,),
 (40,),
 (41,),
 (42,),
 (43,),
 (45,),
 (46,),
 (47,),
 (48,),
 (49,),
 (50,),
 (51,),
 (52,),
 (53,),
 (54,),
 (55,),
 (56,),
 (57,),
 (58,),
 (59,),
 (60,),
 (61,),
 (62,),
 (63,),
 (64,),
 (65,),
 (69,),
 (70,),
 (71,),
 (72,),
 (73,),
 (74,),
 (75,),
 (76,),
 (77,),
 (78,),
 (79,),
 (80,),
 (81,),
 (82,),
 (84,),
 (85,),
 (86,),
 (87,),
 (88,),
 (89,),
 (90,),
 (91,),
 (92,),
 (93,),
 (94,),
 (95,),
 (96,),
 (97,),
 (98,),
 (99,),
 (100,),
 (101,),
 (102,),
 (103,),
 (104,),
 (105,),
 (106,),
 (107,),
 (108,),
 (109,),
 (110,),
 (111,),
 (112,),
 (113,),
 (114,),
 (115,),
 (116,),
 (117,),
 (118,),
 (119,),
 (120,),
 (124,),
 (125,),
 (128,)]

In [14]:
# Obtain list of images
cursor.execute("ALTER TABLE artworks RENAME COLUMN image TO image_url")
cursor.execute("SELECT image_url FROM artworks")
image_urls = cursor.fetchall()
image_urls

[('http://commons.wikimedia.org/wiki/Special:FilePath/Santo%20Domingo%20de%20Silos%20entronizado%20como%20obispo%2C%20por%20Bartolom%C3%A9%20Bermejo.jpg',),
 ('http://commons.wikimedia.org/wiki/Special:FilePath/Bernat-bermejo%20fernando%20I%20castilla.jpg',),
 ('http://commons.wikimedia.org/wiki/Special:FilePath/B.Bermejo%20ret.St.Domingo%20Silos%20hipotesi%204130.jpg',),
 ('http://commons.wikimedia.org/wiki/Special:FilePath/Goya%20MilkMaid.jpg',),
 ('http://commons.wikimedia.org/wiki/Special:FilePath/Los%20duques%20de%20Osuna%20y%20sus%20hijos.jpg',),
 ('http://commons.wikimedia.org/wiki/Special:FilePath/C%C3%B3micos%20ambulantes.jpg',),
 ('http://commons.wikimedia.org/wiki/Special:FilePath/La%20duquesa%20de%20Alba%20y%20la%20Beata.jpg',),
 ('http://commons.wikimedia.org/wiki/Special:FilePath/Baixeras%20-%20Marina%20de%20matinada.jpg',),
 ('http://commons.wikimedia.org/wiki/Special:FilePath/Baixeras%20-%20pageseta.jpg',),
 ('http://commons.wikimedia.org/wiki/Special:FilePath/Francesc%

In [ ]:
from pywikibot.comms import http
import sqlite3

db_path = r".\artworks.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()


def download_images(artwork_data):
    
    for artwork_id, url_string in artwork_data:
        try:
            encoded_filename = url_string.split('/')[-1]
            filename = urllib.parse.unquote(encoded_filename)
            
            print(f"Downloading: {filename} (ID: {artwork_id})...")
            
            response = http.fetch(url_string)
            time.sleep(5)
            
            if response.status_code == 200:
                cursor.execute(
                    "UPDATE artworks SET image_blob = ? WHERE id = ?",
                    (response.content, artwork_id)
                )
                conn.commit()
            else:
                print(f"Failed: Server returned status {response.status_code}")
        except Exception as e:
            print(f"Error processing {url_string}: {e}")

if __name__ == "__main__":
    # Single query fetches both id and image_url
    cursor.execute("SELECT id, image_url FROM artworks")
    artwork_data = cursor.fetchall()
    
    download_images(artwork_data)
    conn.close()

Downloading: Santo Domingo de Silos entronizado como obispo, por Bartolomé Bermejo.jpg (ID: 0)...
Downloading: Bernat-bermejo fernando I castilla.jpg (ID: 1)...
Downloading: B.Bermejo ret.St.Domingo Silos hipotesi 4130.jpg (ID: 2)...
Downloading: Goya MilkMaid.jpg (ID: 3)...
Downloading: Los duques de Osuna y sus hijos.jpg (ID: 4)...
Downloading: Cómicos ambulantes.jpg (ID: 5)...
Downloading: La duquesa de Alba y la Beata.jpg (ID: 6)...
Downloading: Baixeras - Marina de matinada.jpg (ID: 7)...
Downloading: Baixeras - pageseta.jpg (ID: 8)...
Downloading: Francesc Miralles - Gabrielle.jpg (ID: 9)...
Downloading: Jeanne - Joan Brull i Vinyoles (1863-1912).jpg (ID: 10)...
Downloading: Riallera - Joan Brull i Vinyoles (1863-1912).jpg (ID: 11)...
Downloading: Primavera - Joan Brull i Vinyoles (1863-1912) - 0.jpg (ID: 12)...
Downloading: Seriosa - Joan Brull i Vinyoles (1863-1912).jpg (ID: 13)...
Downloading: Josep Cusachs - regiment en marxa.jpg (ID: 14)...
Downloading: Josep Cusachs - Hipòd

In [21]:
# Check para comprobar que todas las imágenes se han descargado
# Porque la bbdd ahora es demasiado grande para visualizarla
# We've selected the ROWID and the length of the blob for a quick overview
db_path = r"C:\Users\evrio\Desktop\TFG\artworks.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT id, image_blob FROM artworks")
rows = cursor.fetchall()

print(f"{'ID':<5} | {'Size (Bytes)':<15} | {'Header (First 10 bytes)'}")
print("-" * 50)

for rowid, blob in rows:
    if blob:
        # Get the first 10 bytes to check the 'Magic Numbers'
        header = blob[:10]
        print(f"{rowid:<5} | {len(blob):<15} | {header}")
    else:
        print(f"{rowid:<5} | EMPTY BLOB")
        
conn.close()

ID    | Size (Bytes)    | Header (First 10 bytes)
--------------------------------------------------
0     | 2790356         | b'\xff\xd8\xff\xe1 LExif'
1     | 5582278         | b'\xff\xd8\xff\xe0\x00\x10JFIF'
2     | 847481          | b'\xff\xd8\xff\xe0\x00\x10JFIF'
3     | 7274719         | b'\xff\xd8\xff\xe0\x00\x10JFIF'
4     | 1538935         | b'\xff\xd8\xff\xe0\x00\x10JFIF'
5     | 2276047         | b'\xff\xd8\xff\xe0\x00\x10JFIF'
6     | 5999650         | b'\xff\xd8\xff\xe1\x13\x95Exif'
7     | 105432          | b'\xff\xd8\xff\xe0\x00\x10JFIF'
8     | 108905          | b'\xff\xd8\xff\xe0\x00\x10JFIF'
9     | 334671          | b'\xff\xd8\xff\xe0\x00\x10JFIF'
10    | 188405          | b'\xff\xd8\xff\xe1\x00\x18Exif'
11    | 722009          | b'\xff\xd8\xff\xe1\x00\x18Exif'
12    | 730206          | b'\xff\xd8\xff\xe1\x00\x18Exif'
13    | 704954          | b'\xff\xd8\xff\xe1\x00\x18Exif'
14    | 58085           | b"\xff\xd8\xff\xe1'2Exif"
15    | 97253           | b'\xff\xd8\xff\

In [8]:
cursor.execute("PRAGMA table_info(artworks);")
columns = cursor.fetchall()
columns

[(0, 'painting', 'TEXT', 0, None, 0),
 (1, 'creator', 'TEXT', 0, None, 0),
 (2, 'movement', 'TEXT', 0, None, 0),
 (3, 'museum', 'TEXT', 0, None, 0),
 (4, 'image', 'TEXT', 0, None, 0),
 (5, 'image_blob', 'BLOB', 0, None, 0)]

In [11]:
cursor.execute("SELECT painting FROM artworks WHERE movement = 'barroco'")
result = cursor.fetchall()
result
conn.close

<function Connection.close()>

In [ ]:
# Comprobamos si hay campos duplicados
db_path = r".\artworks.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT id, painting FROM artworks ORDER BY painting ASC")
results = cursor.fetchall()
results

[(15, "A l'hipòdrom"),
 (57, 'Alegoría de la Industria'),
 (2, 'Altarpiece of Santo Domingo de Silos'),
 (40, 'Antes del baño'),
 (127, 'Antes del baño'),
 (38, 'Au Moulin de la Galette'),
 (22, 'Autoretrat'),
 (116, 'Autorretrato Joaquín Sorolla'),
 (93, 'Basket and Siphon'),
 (123, 'Basket and Siphon'),
 (23, 'Bullring'),
 (44, 'Cabeza de venado o guitarra'),
 (106, 'Cabeza de venado o guitarra'),
 (35, 'Café des Incohérents'),
 (69, 'Carlos II, con armadura'),
 (59, 'Chicos en la playa'),
 (29, 'Claustre de Tarragona'),
 (109, 'Clotilde sentada en un sofá'),
 (24, 'Club de regates de Barcelona'),
 (19, 'Cordova woman'),
 (26, 'Crepuscle. Île Saint-Louis. París'),
 (124, 'Cristo crucificado'),
 (5, 'Cómicos ambulantes'),
 (85, 'Defensa del parque de artillería de Monteleón'),
 (42, 'Don Diego del Corral y Arellano'),
 (43, 'Doña Antonia de Ipeñarrieta y Galdós y su hijo don Luis'),
 (120, 'El Balandrito'),
 (118, 'El Baño en la Granja'),
 (30, 'El Generalife. Granada'),
 (54, 'El aqu

In [ ]:
# Nos encontramos con que los siguientes cuadros tienen campos duplicados:
# Eliminamos los que tienen el movimiento menos adecuado
# El modernismo catalán se ha cambiado por modernismo a secas
# Los cuadros de Juan Gris se han considerado cubistas por la importancia del autor en el movimiento
raw_db_path = r"C:\Users\ev\artworks_raw.db"
conn = sqlite3.connect(db_path)
db_cursor = conn.cursor()
db_cursor.execute("DELETE FROM artworks WHERE id = 127") # Antes del Baño - modernismo catalán
db_cursor.execute("DELETE FROM artworks WHERE id = 123") # Basket and Siphon - purismo vanguardista
db_cursor.execute("DELETE FROM artworks WHERE id = 44") # Cabeza de Venado - barroco
db_cursor.execute("DELETE FROM artworks WHERE id = 68")# El baño del caballo - arte moderno
db_cursor.execute("DELETE FROM artworks WHERE id = 126")# Joven decadente - modernismo catalán
db_cursor.execute("DELETE FROM artworks WHERE id = 121")# La alfombra azul - purismo vanguardista
db_cursor.execute("DELETE FROM artworks WHERE id = 83")# Recuperación de la isla de San Cristobal - barroco (X2) 
db_cursor.execute("DELETE FROM artworks WHERE id = 122") # Violin and fruit dish - purismo vanguardista
conn.commit()
conn.close()

In [4]:
# Ordenamos los resultados alfabéticamente para comprobar que ya no hay pinturas repetidas
db_path = r".\artworks.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT id, painting FROM artworks ORDER BY painting ASC")
results = cursor.fetchall()
results

[(15, "A l'hipòdrom"),
 (57, 'Alegoría de la Industria'),
 (2, 'Altarpiece of Santo Domingo de Silos'),
 (40, 'Antes del baño'),
 (38, 'Au Moulin de la Galette'),
 (22, 'Autoretrat'),
 (116, 'Autorretrato Joaquín Sorolla'),
 (93, 'Basket and Siphon'),
 (23, 'Bullring'),
 (106, 'Cabeza de venado o guitarra'),
 (35, 'Café des Incohérents'),
 (69, 'Carlos II, con armadura'),
 (59, 'Chicos en la playa'),
 (29, 'Claustre de Tarragona'),
 (109, 'Clotilde sentada en un sofá'),
 (24, 'Club de regates de Barcelona'),
 (19, 'Cordova woman'),
 (26, 'Crepuscle. Île Saint-Louis. París'),
 (124, 'Cristo crucificado'),
 (5, 'Cómicos ambulantes'),
 (85, 'Defensa del parque de artillería de Monteleón'),
 (42, 'Don Diego del Corral y Arellano'),
 (43, 'Doña Antonia de Ipeñarrieta y Galdós y su hijo don Luis'),
 (120, 'El Balandrito'),
 (118, 'El Baño en la Granja'),
 (30, 'El Generalife. Granada'),
 (54, 'El aquelarre'),
 (84, 'El baño del caballo'),
 (53, 'El coloso'),
 (37, 'El garrote vil'),
 (32, 'E

In [9]:
cursor.execute("SELECT count() FROM artworks")
result = cursor.fetchall()
result

[(119,)]